# 04 — Two sources of element data: Mathematica and pymatgen

Every feature is a statistic of an **element property** (notebook 01). The paper took those properties
from Wolfram Mathematica's `ElementData`, which is paid software. To make the pipeline independent of
it, the package can build the same table from [pymatgen](https://pymatgen.org/)'s open element data.
The Mathematica table stays available as an option, to check that we reproduce the paper.

**In short:** the pymatgen pipeline has the 10 `*_MeltingPoint` columns instead of the 10
`*_FusionHeat` columns, and its 10 `*_Valence` columns use a different definition; the other 61 columns
have the same meaning (section "Which columns differ").

### Where the code lives

| What | File |
|---|---|
| Both tables, chosen with `source="pymatgen"` (default) or `"mathematica"` | `load_element_table`, `properties` in [`src/supercon/elements.py`](../src/supercon/elements.py) |
| Features with either table | `featurize(counts, source=...)`, `feature_names(source)` in [`src/supercon/features.py`](../src/supercon/features.py) |
| Pipeline parameter | [`scripts/build_features.py`](../scripts/build_features.py) `--elements {pymatgen,mathematica}` |
| Automated checks | [`tests/test_elements.py`](../tests/test_elements.py) |

```bash
uv run python scripts/build_features.py --dataset published --elements mathematica   # the paper's train.csv
uv run python scripts/build_features.py --dataset published --elements pymatgen      # open-source features
uv run python scripts/build_features.py --dataset cleaned   --elements pymatgen
```

## Running the full pipeline

This notebook explains the code with small examples. To produce all the data files instead, run from
the repository root:

```bash
uv sync
uv run python scripts/run_pipeline.py                   # UCI data + Mathematica table + all feature files
uv run python scripts/run_pipeline.py --no-mathematica  # open-source (pymatgen) element table only
uv run pytest                                           # all checks
```

It downloads the UCI data into `data/raw/`, extracts the author's Mathematica element table into
`data/external/`, and writes `data/processed/features_<dataset>_<elements>.csv` for every combination
of `--dataset {published,cleaned}` and `--elements {pymatgen,mathematica}` (see
[`scripts/run_pipeline.py`](../scripts/run_pipeline.py)). A single variant:
`uv run python scripts/build_features.py --dataset cleaned --elements pymatgen`.

**This notebook's part of the pipeline:** the choice of element table, `--elements pymatgen` (default)
or `--elements mathematica`.

In [1]:
import numpy as np
import pandas as pd

from supercon import feature_names, featurize
from supercon.elements import ELEMENTS, load_element_table, properties
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

pmg = load_element_table("pymatgen")
mma = load_element_table("mathematica")
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

## How each property is obtained from pymatgen

The pymatgen table follows the paper's units and conventions (Table 1 of the paper), so that the two
tables are interchangeable:

| Property (paper) | pymatgen source | Conversion / convention |
|---|---|---|
| AtomicMass (amu) | `atomic_mass` | — |
| FirstIonizationEnergy (kJ/mol) | `ionization_energies[0]` (eV) | × 96.485 |
| AtomicRadius (pm) | `atomic_radius_calculated` (Å), else `atomic_radius` | × 100. La and Ce fall back to the empirical radius: 195 and 185 pm, exactly the author's imputation |
| Density at STP (kg/m³) | `density_of_solid` | gases at 0 °C: ideal-gas density of the molecule (H₂, N₂, O₂, F₂, Cl₂ diatomic); liquids (Hg, Br): atomic mass / molar volume |
| ElectronAffinity (kJ/mol) | `electron_affinity` (eV) | × 96.485; no stable anion (negative in pymatgen) → 0, as in Mathematica; then + 1.5 as in the paper |
| **FusionHeat → MeltingPoint (K)** | `melting_point` | pymatgen has no heat of fusion (no open source has it for oxygen) |
| ThermalConductivity (W/(m·K)) | `thermal_conductivity` | — |
| **Valence** | `common_oxidation_states` | largest absolute common oxidation state (pymatgen has no "valence") |

The two bold rows are **changes of definition**, not just of source. The other six are the same physical
quantity from a different database.

In [2]:
pd.concat({"mathematica": mma, "pymatgen": pmg}, axis=1).loc[["H", "O", "Cu", "Ba", "La", "Hg"]].round(2).T

H        O       Cu       Ba       La        Hg
mathematica AtomicMass                1.01    16.00    63.55   137.33   138.91    200.59
            FirstIonizationEnergy  1311.30  1313.10   745.00   502.50   541.10   1006.00
            AtomicRadius             53.00    48.00   145.00   253.00   195.00    171.00
            Density                   0.09     1.43  8960.00  3510.00  6146.00  13534.00
            ElectronAffinity         74.30   142.50   119.90    15.45    49.50      1.50
            FusionHeat                0.56     0.22    13.10     8.00     6.30      2.29
            ThermalConductivity       0.18     0.03   400.00    18.00    13.00      8.30
            Valence                   1.00     2.00     2.00     2.00     3.00      2.00
pymatgen    AtomicMass                1.01    16.00    63.55   137.33   138.91    200.59
            FirstIonizationEnergy  1312.05  1313.94   745.48   502.85   538.09   1007.07
            AtomicRadius             53.00    48.00   145.00   253.00   195.00    171.00
            Density                   0.09     1.43  8920.00  3510.00  6146.00  14236.34
            ElectronAffinity         74.27   142.48   120.74    15.45    55.30      1.50
            MeltingPoint             14.01    54.80  1357.77  1000.00  1193.00    234.32
            ThermalConductivity       0.18     0.03   400.00    18.00    13.00      8.30
            Valence                   1.00     2.00     2.00     2.00     3.00      2.00

## Which columns differ between the two pipelines

Both pipelines produce 81 feature columns in the same order. Column by column they fall into four
categories:

| Category | Columns | What differs |
|---|---|---|
| **Only in the pymatgen pipeline** | 10 × `*_MeltingPoint` | new property (K), replaces FusionHeat |
| **Only in the Mathematica pipeline** | 10 × `*_FusionHeat` | heat of fusion (kJ/mol), not available in pymatgen |
| **Same name, different definition** | 10 × `*_Valence` | Mathematica: typical number of bonds; pymatgen: largest absolute common oxidation state |
| **Same name, same definition** | 51 (`number_of_elements` + 10 × each of `atomic_mass`, `fie`, `atomic_radius`, `Density`, `ElectronAffinity`, `ThermalConductivity`) | only the database the values come from (see the next section for how close they are) |

The renamed columns keep their position: column *k* of one pipeline corresponds to column *k* of the other.
The cell below derives the same classification from the code, and lists the column names.

In [3]:
STATUS = {
    "MeltingPoint": "different property: MeltingPoint (pymatgen) vs FusionHeat (Mathematica)",
    "Valence": "same name, different definition",
}

def status(pmg_col, mma_col):
    if pmg_col != mma_col:
        return STATUS["MeltingPoint"]
    return STATUS["Valence"] if pmg_col.endswith("_Valence") else "same name, same definition"

columns = pd.DataFrame({
    "pymatgen column": feature_names("pymatgen"),
    "mathematica column": feature_names("mathematica"),
})
columns["category"] = [status(p, m) for p, m in zip(columns["pymatgen column"], columns["mathematica column"])]
columns.index = range(1, 82)
columns.index.name = "position"

only_pmg = [c for c in feature_names("pymatgen") if c not in feature_names("mathematica")]
only_mma = [c for c in feature_names("mathematica") if c not in feature_names("pymatgen")]
print("Only in the pymatgen pipeline   :", ", ".join(only_pmg))
print("Only in the Mathematica pipeline:", ", ".join(only_mma))
print("Same name, different definition :", ", ".join(columns.loc[columns["category"] == STATUS["Valence"], "pymatgen column"]))
print()
columns["category"].value_counts().rename("number of columns").to_frame()

Only in the pymatgen pipeline   : mean_MeltingPoint, wtd_mean_MeltingPoint, gmean_MeltingPoint, wtd_gmean_MeltingPoint, entropy_MeltingPoint, wtd_entropy_MeltingPoint, range_MeltingPoint, wtd_range_MeltingPoint, std_MeltingPoint, wtd_std_MeltingPoint
Only in the Mathematica pipeline: mean_FusionHeat, wtd_mean_FusionHeat, gmean_FusionHeat, wtd_gmean_FusionHeat, entropy_FusionHeat, wtd_entropy_FusionHeat, range_FusionHeat, wtd_range_FusionHeat, std_FusionHeat, wtd_std_FusionHeat
Same name, different definition : mean_Valence, wtd_mean_Valence, gmean_Valence, wtd_gmean_Valence, entropy_Valence, wtd_entropy_Valence, range_Valence, wtd_range_Valence, std_Valence, wtd_std_Valence



,number of columns
category,
"same name, same definition",61
different property: MeltingPoint (pymatgen) vs FusionHeat (Mathematica),10
"same name, different definition",10


## How close are the two tables?

For each property: how many of the 86 elements agree within 2%, the correlation over elements (on a
log scale, since values span orders of magnitude), and the elements that differ most.

In [4]:
rows = []
for p_mma, p_pmg in zip(properties("mathematica"), properties("pymatgen")):
    a, b = mma[p_mma], pmg[p_pmg]
    both = a.notna() & b.notna()
    rel = ((b - a).abs() / a.abs()).where(both)
    rows.append({
        "property": p_mma if p_mma == p_pmg else f"{p_mma} vs {p_pmg}",
        "agree within 2%": f"{np.isclose(a, b, rtol=0.02).sum()}/86",
        "corr (log)": round(np.corrcoef(np.log1p(a[both].abs()), np.log1p(b[both].abs()))[0, 1], 3),
        "largest differences": ", ".join(f"{s} {a[s]:.3g}→{b[s]:.3g}" for s in rel.nlargest(3).index) if p_mma == p_pmg else "different quantity",
        "missing (pymatgen)": ", ".join(b.index[b.isna()]) or "—",
    })
pd.DataFrame(rows).set_index("property")

,agree within 2%,corr (log),largest differences,missing (pymatgen)
property,,,,
AtomicMass,86/86,1.000,"Rn 222→220, Po 209→210, Zn 65.4→65.4",—
FirstIonizationEnergy,80/86,0.999,"Hf 575→659, Ta 760→728, Zr 660→640",—
AtomicRadius,84/86,0.998,"Ho 226→175, S 87→88, Kr 87→88",At
Density,82/86,1.000,"Br 3.12e+03→4.04e+03, Hg 1.35e+04→1.42e+04, Nd 7.01e+03→6.8e+03",At
ElectronAffinity,56/86,0.911,"Hf 1.5→18.7, Yb 51.5→1.5, Tm 51.5→101",—
FusionHeat vs MeltingPoint,0/86,0.876,different quantity,—
ThermalConductivity,85/86,1.000,"Pd 71→72, Pt 71→72, Mn 7.7→7.8",—
Valence,70/86,0.814,"Rn 6→0, At 7→1, Mn 4→7",—


* **Atomic mass, radius, density, thermal conductivity, ionization energy**: essentially the same data.
  Where they differ, it is not always pymatgen that is off: Mathematica's first ionization energy of
  hafnium (575 kJ/mol) is wrong; the accepted value (NIST, 6.825 eV) is 658.5 kJ/mol, as in pymatgen.
  The largest density differences are the two liquids, Hg and Br, whose pymatgen value comes from the
  solid's molar volume.
* **Electron affinity**: pymatgen has measured values for the lanthanides, where Mathematica uses a flat
  placeholder of 50 kJ/mol (+1.5).
* **Valence**: the definitions coincide for most main-group elements and differ for several transition
  and noble metals:

In [5]:
v = pd.DataFrame({"mathematica Valence": mma["Valence"], "pymatgen max |ox. state|": pmg["Valence"]})
print(f"identical for {(v.iloc[:, 0] == v.iloc[:, 1]).sum()} of 86 elements; differences:")
v[v.iloc[:, 0] != v.iloc[:, 1]].astype(int).T

identical for 70 of 86 elements; differences:


,N,Cl,Mn,Co,Tc,Ru,Rh,Pr,Os,Ir,Pt,Au,Bi,Po,At,Rn
mathematica Valence,3,5,4,4,6,6,6,4,6,6,6,5,5,6,7,6
pymatgen max |ox. state|,5,7,7,3,7,4,3,3,4,4,4,3,3,4,1,0


* **FusionHeat vs MeltingPoint** are different quantities, but closely related: by Richards' rule the heat
  of fusion of a metal is roughly its melting point times a nearly constant entropy of fusion.

In [6]:
both = mma["FusionHeat"].notna() & pmg["MeltingPoint"].notna()
r = np.corrcoef(np.log(mma.loc[both, "FusionHeat"]), np.log(pmg.loc[both, "MeltingPoint"]))[0, 1]
print(f"correlation of log(FusionHeat) and log(MeltingPoint) over {both.sum()} elements: {r:.2f}")

correlation of log(FusionHeat) and log(MeltingPoint) over 86 elements: 0.93


## Effect on the 81 features

What matters for the models is the features. We compute both versions for the 21,263 published
materials and compare them feature by feature, grouped by property.

In [7]:
unique_m = pd.read_csv(ROOT / "data" / "raw" / "unique_m.csv")
X_mma = featurize(unique_m[ELEMENTS], source="mathematica")
X_pmg = featurize(unique_m[ELEMENTS], source="pymatgen")
print("non-finite values with pymatgen:", (~np.isfinite(X_pmg.to_numpy())).sum())

rows = []
for a, b in zip(feature_names("mathematica"), feature_names("pymatgen")):
    group = "number_of_elements" if a == "number_of_elements" else a.replace("wtd_", "").split("_", 1)[1]
    if a != b:
        group = "FusionHeat -> MeltingPoint"
    rows.append({"property": group, "corr": np.corrcoef(X_mma[a], X_pmg[b])[0, 1],
                 "within 1%": np.isclose(X_mma[a], X_pmg[b], rtol=0.01).mean()})
summary = pd.DataFrame(rows).groupby("property", sort=False).agg(
    corr_min=("corr", "min"), corr_median=("corr", "median"), share_within_1pct=("within 1%", "median"))
summary.round(3)

non-finite values with pymatgen: 0


,corr_min,corr_median,share_within_1pct
property,,,
number_of_elements,1.000,1.000,1.000
atomic_mass,1.000,1.000,1.000
fie,0.998,1.000,0.924
atomic_radius,0.998,1.000,0.989
Density,0.999,1.000,0.952
ElectronAffinity,0.953,0.981,0.488
FusionHeat -> MeltingPoint,0.452,0.866,0.013
ThermalConductivity,1.000,1.000,0.981
Valence,0.741,0.949,0.711


### Column by column

The full list of the 81 columns, their category, and the correlation between the two pipelines over the
21,263 materials (1 = identical up to a linear rescaling). This is the table to use when a result
refers to a specific column.

In [8]:
per_column = columns.copy()
per_column["corr"] = [np.corrcoef(X_pmg[p], X_mma[m])[0, 1] for p, m in
                      zip(columns["pymatgen column"], columns["mathematica column"])]
per_column["within 1%"] = [np.isclose(X_pmg[p], X_mma[m], rtol=0.01).mean() for p, m in
                           zip(columns["pymatgen column"], columns["mathematica column"])]
with pd.option_context("display.max_rows", 100):
    display(per_column.round(3))

,pymatgen column,mathematica column,category,corr,within 1%
position,,,,,
1,number_of_elements,number_of_elements,"same name, same definition",1.000,1.000
2,mean_atomic_mass,mean_atomic_mass,"same name, same definition",1.000,1.000
3,wtd_mean_atomic_mass,wtd_mean_atomic_mass,"same name, same definition",1.000,1.000
4,gmean_atomic_mass,gmean_atomic_mass,"same name, same definition",1.000,1.000
5,wtd_gmean_atomic_mass,wtd_gmean_atomic_mass,"same name, same definition",1.000,1.000
6,entropy_atomic_mass,entropy_atomic_mass,"same name, same definition",1.000,1.000
7,wtd_entropy_atomic_mass,wtd_entropy_atomic_mass,"same name, same definition",1.000,1.000
8,range_atomic_mass,range_atomic_mass,"same name, same definition",1.000,1.000
9,wtd_range_atomic_mass,wtd_range_atomic_mass,"same name, same definition",1.000,1.000


### Reading the comparison

* Five of the eight properties (atomic mass, ionization energy, radius, density, thermal conductivity)
  give **practically the same features** (correlation ≥ 0.998), so
  results that rest on them should carry over.
* Electron affinity and valence features are **strongly but not perfectly** correlated.
* The MeltingPoint features replace the FusionHeat ones and are **new features**, related but not
  interchangeable.

This is why the pipeline keeps both sources: `--elements mathematica` to confirm we reproduce the paper's
numbers, `--elements pymatgen` for an open-source pipeline. Training the same model on both, and checking
whether the same features come out as important, tells us how much the paper's conclusions depend on
the element database.